# Stage 5: Boosting and k-nearest neighbours

This notebook compares a gradient-boosting model with k-nearest neighbours using the shared feature and preprocessing pipeline. All comparisons use the training split; the final test set remains untouched.

In [ ]:
from pathlib import Path
import json
import sys

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import get_split
from src.evaluate import cv_report, save_result
from src.pipeline import build

REPORTS_DIR = PROJECT_ROOT / 'reports'
BEST_PARAMS_PATH = REPORTS_DIR / 'best_params.json'
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Split first

In [ ]:
X_train, X_test, y_train, y_test = get_split()
print(f'Training rows: {len(X_train):,}')
print(f'Held-out test rows: {len(X_test):,}')
print(f'Training yes-rate: {y_train.mean():.3%}')

### Observation and decision
The split comes before cross-validation and tuning. Every transformer in `build()` is fitted inside each training fold, and the held-out test rows are not used for model selection.

## HistGradientBoosting baseline and tuning

In [ ]:
# HistGradientBoosting is sklearn's fast implementation of the GBM idea from lectures.
hgb_baseline = build(HistGradientBoostingClassifier(class_weight='balanced', random_state=42))
baseline_row = cv_report('HistGradientBoosting', hgb_baseline, X_train, y_train)
save_result(baseline_row, REPORTS_DIR / 'model_comparison.csv')
baseline_row

### Observation and decision
Boosting grows trees in sequence. Each later tree focuses more on examples that the earlier trees predicted poorly, so the combined model gradually corrects earlier errors. The balanced class weight gives the minority yes class more influence during the baseline comparison.

In [ ]:
hgb_search = RandomizedSearchCV(
    estimator=build(HistGradientBoostingClassifier(class_weight='balanced', random_state=42)),
    param_distributions={
        'model__learning_rate': [0.02, 0.05, 0.1, 0.2],
        'model__max_depth': [None, 3, 5, 7],
        'model__max_leaf_nodes': [15, 31, 63, 127],
        'model__min_samples_leaf': [10, 20, 50, 100],
        'model__l2_regularization': [0.0, 0.1, 1.0, 10.0],
    },
    n_iter=25,
    scoring='average_precision',
    cv=cv,
    random_state=42,
    n_jobs=-1,
)
hgb_search.fit(X_train, y_train)
hgb_tuned_row = cv_report('HistGradientBoosting-tuned', hgb_search.best_estimator_, X_train, y_train)
save_result(hgb_tuned_row, REPORTS_DIR / 'model_comparison.csv')
print(hgb_search.best_params_)
hgb_tuned_row

### Observation and decision
The search uses the pipeline's `model__` prefix because the classifier is the `model` step inside `build()`. `average_precision` is the primary score because the subscribed class is uncommon; the other metrics from `cv_report` remain useful supporting evidence.

## k-nearest neighbours baseline and tuning

In [ ]:
# The shared pipeline scales numeric columns before kNN calculates distances.
knn_baseline = build(KNeighborsClassifier(n_neighbors=15, n_jobs=-1))
knn_baseline_row = cv_report('KNeighbors', knn_baseline, X_train, y_train)
save_result(knn_baseline_row, REPORTS_DIR / 'model_comparison.csv')
knn_baseline_row

### Observation and decision
kNN predicts from distances to nearby training examples. Scaling the numeric features in the pipeline matters because a large-scale column such as balance could otherwise dominate the distance. kNN can also be slow at prediction time: for every new client it compares that row with many stored training rows instead of using a compact fitted formula.

In [ ]:
knn_search = GridSearchCV(
    estimator=build(KNeighborsClassifier(n_jobs=-1)),
    param_grid={
        'model__n_neighbors': [5, 15, 31, 51],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2],
    },
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
)
knn_search.fit(X_train, y_train)
knn_tuned_row = cv_report('KNeighbors-tuned', knn_search.best_estimator_, X_train, y_train)
save_result(knn_tuned_row, REPORTS_DIR / 'model_comparison.csv')
print(knn_search.best_params_)
knn_tuned_row

### Observation and decision
The grid compares neighbourhood size, distance weighting, and Manhattan versus Euclidean distance. A small neighbourhood can be sensitive to noise, while a large one can wash out local patterns; cross-validation chooses the trade-off instead of guessing.

## Preserve the best-parameter record

In [ ]:
# Update the JSON object so teammate entries are retained.
if BEST_PARAMS_PATH.exists():
    best_params = json.loads(BEST_PARAMS_PATH.read_text())
else:
    best_params = {}
best_params['HistGradientBoosting-tuned'] = hgb_search.best_params_
best_params['KNeighbors-tuned'] = knn_search.best_params_
BEST_PARAMS_PATH.write_text(json.dumps(best_params, indent=2) + '\n')
print(json.dumps(best_params, indent=2))

### Observation and decision
The baseline and tuned rows are saved with distinct names, so the comparison report can show whether tuning helped. The best-parameter file is loaded, updated, and written back, preserving the existing DecisionTree and RandomForest entries.

## Final student-level takeaway

HistGradientBoosting is scikit-learn's fast implementation of gradient boosting, the GBM idea from our lectures: later trees concentrate on errors left by earlier trees. kNN is easy to understand, but it needs scaled numeric inputs because distance is its whole decision rule, and it can be slow when predicting because it searches through the stored training examples. We will choose between the tuned candidates using cross-validated PR-AUC first, then use recall, precision, and ROC-AUC to understand the practical trade-off.